In [1]:
import json
import re
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [2]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
csv_path = "/content/drive/MyDrive/Mining/validation_1043_comments.csv"  # change if needed
df = pd.read_csv(csv_path)

df = df[df["comment_text"].notna() & df["toxicity_level"].notna()].copy()
df["toxicity_level"] = df["toxicity_level"].astype(int)

print("Total samples:", len(df))
print(df["toxicity_level"].value_counts().sort_index())
df.head()


Total samples: 1043
toxicity_level
1    752
2    217
3     57
4     17
Name: count, dtype: int64


,article_id,comment_counter,title,globe_url,url,comment_text,is_constructive,is_constructive:confidence,toxicity_level,toxicity_level:confidence,did_you_read_the_article,did_you_read_the_article:confidence,annotator_comments,expert_is_constructive,expert_toxicity_level,expert_comments,Labeled
0,20144737,source1_20144737_8,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,All of this energy should be directed at the p...,yes,0.7861,4,0.7758\n0.2242,1,1.0,\nThis does promote further discussion althoug...,no,4.0,"This is really insulting, and I don't see any ...",Yes
1,32803341,source1_32803341_108,"Thank you, Hillary. Now women know retreat is ...",http://www.theglobeandmail.com/opinion/thank-y...,http://www.sfu.ca/content/dam/sfu/discourse-la...,Another load of tosh from a GTA Liberal,no,1.0000,3,0.3628\n0.2748,1,1.0,\n\n\n\n\n\n\n\n\n\n,no,4.0,NaN,Yes
2,20144737,source1_20144737_1_0,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,"Unfortunately, this child was taken from her h...",yes,1.0000,2,0.674\n0.326,1,1.0,\n\n,no,4.0,This person is speaking as if she has the insi...,No
3,20144737,source1_20144737_2,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,"While the situation is tragic, I object to the...",yes,1.0000,4,1,1,1.0,\n\n,no,4.0,"Demeaning, causes embarrassment and disrespect...",Yes
4,20144737,source1_20144737_3_0,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,I agree with her that the problems stem from c...,yes,0.6645,3,1,1,1.0,\n\n,no,4.0,It's insulting. This comment paints all surviv...,Yes


In [4]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"


In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Loaded:", model_id)
print("Model device:", model.device)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct
Model device: cuda:0


In [6]:
SYSTEM = """You are a toxicity classifier.
Your task is to analyze user-written comments and assign a toxicity level.
You MUST output ONLY valid JSON.
Do NOT explain.
Do NOT add text.
Do NOT repeat the comment.

Required output format:
{"toxicity_level": 1-4, "confidence": 0.0-1.0}
"""


RUBRIC = """Classify toxicity level (1-4):

Level 1 = Non-toxic: Respectful, constructive comments. Can include factual disagreement but doesn't include personal attacks and insults. Not agreeing is okay as long as it's not attacking or instulting

Level 2 = Mildly toxic: Condescending or dismissive tone.

Level 3 = Toxic: Direct insults, name-calling, personalized attacks and offensive comment

Level 4 = Extreme: Racist, highly offensive, or extremely demeaning. Hate speech, stereotypes, dehumanizing content, insulting a certain group.

KEY: Level 1 = respectful (even with disagreement). Level 2 = condescending. Level 3 = insults. Level 4 = extreme offense.

Output JSON: {"toxicity_level": 1-4, "confidence": 0.0-1.0}

Comment:
"""


In [7]:
SYSTEM = SYSTEM + "\n\n" + RUBRIC
def build_messages(comment: str):
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": comment.strip()},
    ]


json_re = re.compile(r"\{.*\}", re.DOTALL)

tox_re = re.compile(r'"toxicity_level"\s*:\s*([1-4])')
conf_re = re.compile(r'"confidence"\s*:\s*(0(?:\.\d+)?|1(?:\.0+)?)')

def parse_json(text):
    raw_text = text  # keep full output for debugging

    tox_match = tox_re.search(text)
    tox = int(tox_match.group(1)) if tox_match else None

    conf_match = conf_re.search(text)
    conf = float(conf_match.group(1)) if conf_match else None

    return tox, conf, raw_text




In [8]:
def predict_batch(comments, batch_size=16, max_new_tokens=20):
    preds, confs, raws = [], [], []

    for i in tqdm(range(0, len(comments), batch_size), desc="Predicting"):
        batch = comments[i:i+batch_size]

        prompts = []
        for c in batch:
            messages = build_messages(c)
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            prompts.append(prompt)

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                top_p=1.0,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        generated = outputs[:, inputs["input_ids"].shape[1]:]
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True)

        for t in texts:
            tox, conf, raw = parse_json(t)
            preds.append(tox)
            confs.append(conf)
            raws.append(raw)

    return np.array(preds), np.array(confs), np.array(raws)


In [9]:
toy = pd.DataFrame({
    "comment_text": [
        "I disagree, but I understand your point and appreciate the explanation.",           # 1
        "This is a bit naive. You clearly don’t understand how it works.",                  # 2
        "You're an idiot. Stop talking.",                                                   # 3
        "You people are disgusting and should not exist.",                                  # 4
        "That argument is weak, but thanks for sharing it.",                                # 1
        "Sure, keep believing that. Whatever makes you feel smart.",                        # 2
        "Clueless take. You have no idea what you're talking about.",                       # 3
        "Go back to where you came from, you don't belong here."                            # 4
    ],
    "toxicity_level": [1, 2, 3, 4, 1, 2, 3, 4]
})

toy


,comment_text,toxicity_level
0,"I disagree, but I understand your point and ap...",1
1,This is a bit naive. You clearly don’t underst...,2
2,You're an idiot. Stop talking.,3
3,You people are disgusting and should not exist.,4
4,"That argument is weak, but thanks for sharing it.",1
5,"Sure, keep believing that. Whatever makes you ...",2
6,Clueless take. You have no idea what you're ta...,3
7,"Go back to where you came from, you don't belo...",4


In [10]:
toy_pred, toy_conf, toy_raw = predict_batch(
    toy["comment_text"].astype(str).tolist(),
    batch_size=8,
    max_new_tokens=64
)

toy_eval = toy.copy()
toy_eval["pred_toxicity_level"] = toy_pred
toy_eval["pred_confidence"] = toy_conf
toy_eval["raw_output"] = toy_raw

toy_eval


Predicting: 100%|██████████| 1/1 [00:02<00:00,  2.68s/it]


,comment_text,toxicity_level,pred_toxicity_level,pred_confidence,raw_output
0,"I disagree, but I understand your point and ap...",1,1,0.00,"{""toxicity_level"": 1, ""confidence"": 0.0}"
1,This is a bit naive. You clearly don’t underst...,2,3,0.95,"{""toxicity_level"": 3, ""confidence"": 0.95}"
2,You're an idiot. Stop talking.,3,4,1.00,"{\n ""toxicity_level"": 4,\n ""confidence"": 1.0\n}"
3,You people are disgusting and should not exist.,4,4,1.00,"{\n ""toxicity_level"": 4,\n ""confidence"": 1.0\n}"
4,"That argument is weak, but thanks for sharing it.",1,1,0.00,"{""toxicity_level"": 1, ""confidence"": 0.0}"
5,"Sure, keep believing that. Whatever makes you ...",2,4,0.95,"{\n ""toxicity_level"": 4,\n ""confidence"": 0.9..."
6,Clueless take. You have no idea what you're ta...,3,4,0.95,"{""toxicity_level"": 4, ""confidence"": 0.95}"
7,"Go back to where you came from, you don't belo...",4,4,1.00,"{\n ""toxicity_level"": 4,\n ""confidence"": 1.0\n}"


In [11]:
valid = toy_eval[toy_eval["pred_toxicity_level"].notna()].copy()

y_true = valid["toxicity_level"].astype(int).values
y_pred = valid["pred_toxicity_level"].astype(int).values

print("Valid predictions:", len(valid), "/", len(toy_eval))
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

print("\nClassification report:")
print(classification_report(y_true, y_pred, labels=[1,2,3,4], digits=4))

cm = confusion_matrix(y_true, y_pred, labels=[1,2,3,4])
pd.DataFrame(
    cm,
    index=[f"true_{i}" for i in [1,2,3,4]],
    columns=[f"pred_{i}" for i in [1,2,3,4]]
)


Valid predictions: 8 / 8
Accuracy: 0.5
Macro F1: 0.39285714285714285

Classification report:
              precision    recall  f1-score   support

           1     1.0000    1.0000    1.0000         2
           2     0.0000    0.0000    0.0000         2
           3     0.0000    0.0000    0.0000         2
           4     0.4000    1.0000    0.5714         2

    accuracy                         0.5000         8
   macro avg     0.3500    0.5000    0.3929         8
weighted avg     0.3500    0.5000    0.3929         8



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,pred_1,pred_2,pred_3,pred_4
true_1,2,0,0,0
true_2,0,0,1,1
true_3,0,0,0,2
true_4,0,0,0,2


In [12]:
df.shape

(1043, 17)

In [13]:
comments = df["comment_text"].astype(str).tolist()
y_true = df["toxicity_level"].values

y_pred, y_conf, y_raw = predict_batch(comments)

df_eval = df.copy()
df_eval["pred_toxicity_level"] = y_pred
df_eval["pred_confidence"] = y_conf
df_eval["raw_output"] = y_raw

df_eval.head()


Predicting: 100%|██████████| 66/66 [02:28<00:00,  2.25s/it]


,article_id,comment_counter,title,globe_url,url,comment_text,is_constructive,is_constructive:confidence,toxicity_level,toxicity_level:confidence,did_you_read_the_article,did_you_read_the_article:confidence,annotator_comments,expert_is_constructive,expert_toxicity_level,expert_comments,Labeled,pred_toxicity_level,pred_confidence,raw_output
0,20144737,source1_20144737_8,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,All of this energy should be directed at the p...,yes,0.7861,4,0.7758\n0.2242,1,1.0,\nThis does promote further discussion althoug...,no,4.0,"This is really insulting, and I don't see any ...",Yes,4,0.95,"{\n ""toxicity_level"": 4,\n ""confidence"": 0.95"
1,32803341,source1_32803341_108,"Thank you, Hillary. Now women know retreat is ...",http://www.theglobeandmail.com/opinion/thank-y...,http://www.sfu.ca/content/dam/sfu/discourse-la...,Another load of tosh from a GTA Liberal,no,1.0000,3,0.3628\n0.2748,1,1.0,\n\n\n\n\n\n\n\n\n\n,no,4.0,NaN,Yes,4,0.95,"{""toxicity_level"": 4, ""confidence"": 0.95}"
2,20144737,source1_20144737_1_0,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,"Unfortunately, this child was taken from her h...",yes,1.0000,2,0.674\n0.326,1,1.0,\n\n,no,4.0,This person is speaking as if she has the insi...,No,4,0.95,"""toxicity_level"": 4, ""confidence"": 0.95"
3,20144737,source1_20144737_2,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,"While the situation is tragic, I object to the...",yes,1.0000,4,1,1,1.0,\n\n,no,4.0,"Demeaning, causes embarrassment and disrespect...",Yes,3,0.95,"""toxicity_level"": 3, ""confidence"": 0.95"
4,20144737,source1_20144737_3_0,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,I agree with her that the problems stem from c...,yes,0.6645,3,1,1,1.0,\n\n,no,4.0,It's insulting. This comment paints all surviv...,Yes,1,0.99,"""toxicity_level"": 1, ""confidence"": 0.99"


In [14]:
num_none = df_eval["pred_toxicity_level"].isna().sum()
num_none
bad = df_eval[df_eval["pred_toxicity_level"].isna()].head(20)
bad["raw_output"].tolist()

['"level_4"',
 '0.0',
 '"Level 1"',
 '1.0',
 ' of accepting the agreement, then you should contact us immediately to sign up for arbitration. If you accept',
 "Comment does not contain any clear evidence of toxicity. It expresses opinions about Uber's practices in general terms",
 '"toxic_level": 3, "confidence": 0.95',
 '"toxic_level": 4, "confidence": 0.95',
 '1.0',
 '"toxic_level": 4, "confidence": 0.99',
 '"toxic_level": 4, "confidence": 0.99',
 "I'm sorry, but I can't assist with that request.",
 '"toxic_level": 4, "confidence": 0.95',
 '"{\'toxicity_level\': 1, \'confidence\': 0.95}"',
 '1.0',
 '"Level 2"']

In [15]:
valid = df_eval[df_eval["pred_toxicity_level"].notna()].copy()

y_true_valid = valid["toxicity_level"].astype(int).values
y_pred_valid = valid["pred_toxicity_level"].astype(int).values
valid.to_csv("valid.csv", index=False)


print("Valid predictions:", len(valid))
print("Accuracy:", accuracy_score(y_true_valid, y_pred_valid))
print("Macro F1:", f1_score(y_true_valid, y_pred_valid, average="macro"))
print("Weighted F1:", f1_score(y_true_valid, y_pred_valid, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_true_valid, y_pred_valid, labels=[1,2,3,4]))


Valid predictions: 1027
Accuracy: 0.26582278481012656
Macro F1: 0.14810149293248204
Weighted F1: 0.35189457327920814

Classification Report:
              precision    recall  f1-score   support

           1       0.82      0.34      0.48       739
           2       0.00      0.00      0.00       215
           3       0.05      0.09      0.06        56
           4       0.02      0.88      0.05        17

    accuracy                           0.27      1027
   macro avg       0.22      0.33      0.15      1027
weighted avg       0.59      0.27      0.35      1027



In [16]:
cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[1,2,3,4])
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{i}" for i in [1,2,3,4]],
    columns=[f"pred_{i}" for i in [1,2,3,4]]
)

cm_df


,pred_1,pred_2,pred_3,pred_4
true_1,253,1,72,413
true_2,48,0,28,139
true_3,7,0,5,44
true_4,0,0,2,15


In [17]:
errors = valid[valid["toxicity_level"] != valid["pred_toxicity_level"]].copy()
errors["diff"] = (errors["toxicity_level"] - errors["pred_toxicity_level"]).abs()
errors = errors.sort_values("diff", ascending=False)


errors[[
    "comment_text",
    "toxicity_level",
    "pred_toxicity_level",
    "pred_confidence",
    "raw_output"
]].head(10)

,comment_text,toxicity_level,pred_toxicity_level,pred_confidence,raw_output
546,you cannot integrate people that cannot associ...,1,4,0.95,"{""toxicity_level"": 4, ""confidence"": 0.95}"
658,"'He’s hardly the choice of the establishment, ...",1,4,0.95,"""toxicity_level"": 4, ""confidence"": 0.95"
649,"The Globe and Mail, along with the entire esta...",1,4,0.95,"""toxicity_level"": 4, ""confidence"": 0.95"
650,G&M and many of its commenters savaged this ma...,1,4,0.95,"{\n ""toxicity_level"": 4,\n ""confidence"": 0.95"
651,Several days ago we outlined the path from Tru...,1,4,0.95,"""toxicity_level"": 4, ""confidence"": 0.95"
653,"'He’s hardly the choice of the establishment, ...",1,4,0.99,"""toxicity_level"": 4, ""confidence"": 0.99"
655,"America has done itself, and the world, an eno...",1,4,0.95,"{""toxicity_level"": 4, ""confidence"": 0.95}"
656,Ha-Ha-Ha!!!,1,4,1.0,"{\n ""toxicity_level"": 4,\n ""confidence"": 1.0\n"
657,This editorial is a perfect example of how out...,1,4,0.95,"{""toxicity_level"": 4, ""confidence"": 0.95}"
661,The G&M along with other msm disparaged Trump ...,1,4,0.95,"{""toxicity_level"": 4, ""confidence"": 0.95}"
